In [6]:
import IPython
from google.colab import output
import numpy as np

# Global variable: like a shared storage bin in a workshop accessible to everyone
shapes_in_python = []

# Callback function: like leaving your contact number with a receptionist so they notify you when your package arrives
def receive_shapes(shapes_data):
    """Receives shapes sent from JavaScript directly into the Python environment."""
    global shapes_in_python
    shapes_in_python = shapes_data
    print(f"\n[Python Kernel] Successfully synchronized {len(shapes_in_python)} shape(s) from the canvas!")

    for idx, shape in enumerate(shapes_in_python):
        pts = [[p['x'], p['y']] for p in shape]
        print(f" Shape {idx + 1} ({len(pts)} vertices):")

        # Test ShapeTransformer routines
        translated = ShapeTransformer.translate(pts, 30, 20)
        rotated = ShapeTransformer.rotate(pts, np.radians(45))
        # Fixed: pass a single uniform scale factor s = 1.2
        scaled = ShapeTransformer.scale(pts, 1.2)

        print(f"  - Original point 0:     {pts[0]}")
        print(f"  - Translated (+30,+20): {translated[0]}")
        print(f"  - Rotated 45°:          [{rotated[0][0]:.2f}, {rotated[0][1]:.2f}]")
        print(f"  - Uniformly Scaled 1.2x:[{scaled[0][0]:.2f}, {scaled[0][1]:.2f}]")

# Registering the Colab bridge: like installing a mail chute connecting two floors of an office
output.register_callback('notebook.receive_shapes', receive_shapes)

# HTML and JavaScript front-end
html_code = """
<!DOCTYPE html>
<html>
<head>
<title>2D Shape Transformation Tool</title>
<style>
  body { font-family: Arial, sans-serif; margin: 10px; }
  /* Canvas element: like a blank drafting board mounted on a studio wall */
  canvas { border: 2px solid #333; margin: 10px 0; background: #fff; cursor: crosshair; }
  .controls { margin-bottom: 8px; }
  button {
    padding: 6px 12px;
    margin-right: 6px;
    cursor: pointer;
    border: 1px solid #777;
    border-radius: 4px;
    background-color: #f0f0f0;
  }
  button.active {
    background-color: #007bff;
    color: white;
    font-weight: bold;
  }
  #syncBtn { background-color: #28a745; color: white; border: none; font-weight: bold; }
  #syncBtn:hover { background-color: #218838; }
  #status { font-size: 13px; color: #555; margin-top: 4px; }
</style>
</head>
<body>

<div class="controls">
  <button id="drawBtn" class="active">Draw Mode</button>
  <button id="translateBtn">Translate Mode</button>
  <button id="rotateBtn">Rotate Mode</button>
  <button id="scaleBtn">Scale Mode</button>
  <button id="resetBtn">Reset</button>
  <button id="syncBtn">Send to Python</button>
</div>

<div id="status">Mode: Draw. Click to plot points; click within 10px of the first point to close the polygon.</div>
<canvas id="myCanvas" width="500" height="300"></canvas>

<script>
const canvas = document.getElementById('myCanvas');
const ctx = canvas.getContext('2d');
const statusDiv = document.getElementById('status');

let shapes = [];
let currentShape = [];
let isDrawing = false;
let transformMode = null; // null = Draw mode
let selectedShapeIndex = -1;

let startX, startY;
let originalPoints = [];

// UI Mode button handling
const buttons = {
  draw: document.getElementById('drawBtn'),
  translate: document.getElementById('translateBtn'),
  rotate: document.getElementById('rotateBtn'),
  scale: document.getElementById('scaleBtn')
};

function setActiveMode(mode) {
  transformMode = mode;
  Object.values(buttons).forEach(btn => btn.classList.remove('active'));
  if (mode === null) {
    buttons.draw.classList.add('active');
    statusDiv.innerText = 'Mode: Draw. Click to add vertices; click near the red start point to close.';
  } else {
    buttons[mode].classList.add('active');
    statusDiv.innerText = `Mode: ${mode.toUpperCase()}. Click and drag near any shape vertex to transform it.`;
  }
}

buttons.draw.addEventListener('click', () => setActiveMode(null));
buttons.translate.addEventListener('click', () => setActiveMode('translate'));
buttons.rotate.addEventListener('click', () => setActiveMode('rotate'));
buttons.scale.addEventListener('click', () => setActiveMode('scale'));
document.getElementById('resetBtn').addEventListener('click', resetCanvas);

// Send shapes to Python kernel via Colab output bridge
document.getElementById('syncBtn').addEventListener('click', () => {
  if (shapes.length === 0) {
    alert('Draw and close at least one shape before sending to Python!');
    return;
  }
  google.colab.kernel.invokeFunction('notebook.receive_shapes', [shapes], {});
  statusDiv.innerText = `Synchronized ${shapes.length} shape(s) with Python! Check cell output below.`;
});

// Canvas Mouse Events
// An event listener: like a doorbell waiting silently for someone to press it
canvas.addEventListener('mousedown', handleMouseDown);
canvas.addEventListener('mousemove', handleMouseMove);
canvas.addEventListener('mouseup', handleMouseUp);

function handleMouseDown(e) {
  const rect = canvas.getBoundingClientRect();
  const x = e.clientX - rect.left;
  const y = e.clientY - rect.top;

  if (!transformMode) {
    // Drawing mode
    if (!isDrawing) {
      currentShape = [{x, y}];
      isDrawing = true;
      drawShapes();
      drawPoint(x, y, 'red');
    } else {
      const firstPoint = currentShape[0];
      // Euclidean distance: like stretching a measuring tape directly between two pins on a board
      const distance = Math.sqrt((x - firstPoint.x) ** 2 + (y - firstPoint.y) ** 2);

      if (distance <= 10 && currentShape.length >= 2) {
        currentShape.push({x: firstPoint.x, y: firstPoint.y});
        shapes.push([...currentShape]);
        currentShape = [];
        isDrawing = false;
        drawShapes();
      } else {
        currentShape.push({x, y});
        drawShapes();
      }
    }
  } else {
    // Transformation selection
    startX = x;
    startY = y;
    selectedShapeIndex = findShapeAtPosition(x, y);
    if (selectedShapeIndex >= 0) {
      originalPoints = JSON.parse(JSON.stringify(shapes[selectedShapeIndex]));
    }
  }
}

function handleMouseMove(e) {
  if (!transformMode || selectedShapeIndex < 0) return;

  const rect = canvas.getBoundingClientRect();
  const x = e.clientX - rect.left;
  const y = e.clientY - rect.top;

  const dx = x - startX;
  const dy = y - startY;

  if (transformMode === 'translate') {
    // Translation: like sliding a dinner plate across a smooth table without spinning it
    shapes[selectedShapeIndex] = originalPoints.map(p => ({
      x: p.x + dx,
      y: p.y + dy
    }));
  }
  else if (transformMode === 'rotate') {
    // Centroid: like the exact point where a cardboard cutout balances on a fingertip
    const center = getShapeCenter(originalPoints);
    // Rotation: like turning a steering wheel around its center column
    const angle = Math.atan2(y - center.y, x - center.x) - Math.atan2(startY - center.y, startX - center.x);

    shapes[selectedShapeIndex] = originalPoints.map(p => {
      const tx = p.x - center.x;
      const ty = p.y - center.y;
      return {
        x: tx * Math.cos(angle) - ty * Math.sin(angle) + center.x,
        y: tx * Math.sin(angle) + ty * Math.cos(angle) + center.y
      };
    });
  }
  else if (transformMode === 'scale') {
    const center = getShapeCenter(originalPoints);

    // Uniform scale factor: like adjusting the percentage zoom on a projector so height and width grow equally
    const scale = Math.max(0.1, 1 + dx / 100);

    // Applied 'scale' uniformly to both horizontal and vertical axes
    shapes[selectedShapeIndex] = originalPoints.map(p => ({
      x: center.x + (p.x - center.x) * scale,
      y: center.y + (p.y - center.y) * scale
    }));
  }

  drawShapes();
}

function handleMouseUp() {
  selectedShapeIndex = -1;
}

function findShapeAtPosition(x, y) {
  for (let i = 0; i < shapes.length; i++) {
    for (const pt of shapes[i]) {
      if (Math.sqrt((pt.x - x) ** 2 + (pt.y - y) ** 2) < 15) {
        return i;
      }
    }
  }
  return -1;
}

function getShapeCenter(points) {
  const sum = points.reduce((acc, p) => {
    acc.x += p.x;
    acc.y += p.y;
    return acc;
  }, {x: 0, y: 0});

  return {
    x: sum.x / points.length,
    y: sum.y / points.length
  };
}

function drawPoint(x, y, color) {
  ctx.fillStyle = color;
  ctx.fillRect(x - 2, y - 2, 4, 4);
}

function drawShapes() {
  ctx.clearRect(0, 0, canvas.width, canvas.height);

  // Render completed shapes
  shapes.forEach(shape => {
    shape.forEach(p => drawPoint(p.x, p.y, 'red'));

    ctx.beginPath();
    ctx.moveTo(shape[0].x, shape[0].y);
    for (let i = 1; i < shape.length; i++) {
      ctx.lineTo(shape[i].x, shape[i].y);
    }
    ctx.closePath();
    ctx.strokeStyle = 'blue';
    ctx.stroke();
  });

  // Render current shape under construction
  if (currentShape.length > 0) {
    currentShape.forEach(p => drawPoint(p.x, p.y, 'red'));

    ctx.beginPath();
    ctx.moveTo(currentShape[0].x, currentShape[0].y);
    for (let i = 1; i < currentShape.length; i++) {
      ctx.lineTo(currentShape[i].x, currentShape[i].y);
    }
    ctx.strokeStyle = 'green';
    ctx.stroke();
  }
}

function resetCanvas() {
  shapes = [];
  currentShape = [];
  isDrawing = false;
  setActiveMode(null);
  ctx.clearRect(0, 0, canvas.width, canvas.height);
}
</script>
</body>
</html>
"""

# Render the interactive canvas UI
display(IPython.display.HTML(html_code))

# Python Transformation Module
class ShapeTransformer:
    """Mathematical 2D transformations on vertex sets."""

    # Static method: like an independent hand calculator that performs math without needing external setup
    @staticmethod
    def translate(points, dx, dy):
        """Translates coordinate points by offset (dx, dy)."""
        return [[p[0] + dx, p[1] + dy] for p in points]

    @staticmethod
    def rotate(points, angle_rad, center=None):
        """Rotates points around a centroid or specified origin by an angle in radians."""
        if center is None:
            center = np.mean(points, axis=0)
        cx, cy = center
        cos_v, sin_v = np.cos(angle_rad), np.sin(angle_rad)
        return [
            [
                cx + (p[0] - cx) * cos_v - (p[1] - cy) * sin_v,
                cy + (p[0] - cx) * sin_v + (p[1] - cy) * cos_v
            ]
            for p in points
        ]

    @staticmethod
    def scale(points, s, center=None):
        """Uniformly scale points around a center by a single factor s."""
        if center is None:
            center = np.mean(points, axis=0)
        cx, cy = center
        return [
            [cx + (p[0] - cx) * s, cy + (p[1] - cy) * s]
            for p in points
        ]

Modify the original code to achieve uniform scaling.

Step 1: Modify the JavaScript code for the scale function. Delete "sx" and "sy", and use the "scale" variable instead. Then, apply the same scale factor to both the x and y axes.

Step 2: Modify the Python code. Change the parameters "sx" and "sy" to "s", then replace them with "s" in the list comprehension.
